# Gerar Legenda MULTICOR — 5 Idiomas (classificação gramatical por palavra)

Gera o arquivo `.ass` com a legenda colorida — não queima em nenhum vídeo
ainda (isso é o notebook `caption-multicolor-burn.ipynb`, separado de
propósito, pra dar espaço pra correção manual no meio do caminho).

No final, duas ações **separadas**: baixar o `.ass` pra revisar/corrigir,
e (só depois de confirmar que ficou bom) salvar no Drive.


---

> ### 🇨🇳 Variante de 6 idiomas (com chinês)
> Cópia do notebook de 5 idiomas com o chinês (`zh`) incluído — o original
> continua intacto e funcionando. As duas variantes convivem na mesma pasta
> do vídeo: esta grava os finais com sufixo `_zh`
> (`<nome>_final_idiomas_zh.mp4`, `<nome>_final_multicolor_zh.mp4`), então
> rodar esta **não** sobrescreve o resultado de 5 idiomas.
>
> Convenção de código do chinês: `zh` internamente (posição na tela, cor,
> fonte) · `zh-Hans` no YouTube · `zh-hans` no Stanza. Simplificado, não
> tradicional — é o usado na China continental, Singapura e Malásia.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q stanza kiwipiepy

import shutil, sys
from pathlib import Path

import stanza
from kiwipiepy import Kiwi
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")

# ── O Drive tem TODOS os módulos? ──────────────────────────────────────────
# "N módulos copiados" sozinho não quer dizer nada: um Drive com 13 dos 31
# saía com visto verde, e o notebook quebrava depois num import, longe da
# causa. O manifesto (gravado pelo repositorio-sincronizar) diz quantos
# DEVIAM estar lá.
_manifesto = PASTA_MODULOS / "_manifesto.txt"
if _manifesto.exists():
    _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                  if l.strip() and not l.startswith("#")}
    _faltando = sorted(_esperados - {f.name for f in DESTINO.glob("*.py")})
    if _faltando:
        print(f"\n🚨 FALTAM {len(_faltando)} de {len(_esperados)} módulos no Drive:")
        for _n in _faltando:
            print(f"     {_n}")
        raise SystemExit(
            "Rode o repositorio-sincronizar.ipynb antes de continuar. "
            "Seguir assim quebra num import lá na frente, longe da causa.")
    print(f"   ✅ os {len(_esperados)} módulos do manifesto estão presentes")
else:
    print("   ⚠️  sem _manifesto.txt — rode o repositorio-sincronizar pra criá-lo")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ⚠️ Os módulos abaixo (classificacao.py, classificacao_ko.py, cores.py,
# renderizacao.py) precisam estar na MESMA pasta
# narrated_video/pipeline/modulos/ do Drive, junto com os antigos, pra esse
# copytree acima já trazer eles também. Se der erro de import na célula 4,
# é sinal de que faltou subir algum desses 4 arquivos pro Drive.

kiwi = Kiwi()
print("✅ Stanza e Kiwi prontos")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"

IDIOMA_MESTRE = "en"  # idioma de referência — usa o SRT "whisper" (bruto, não
                      # sincronizado); os outros idiomas usam o conjunto já
                      # sincronizado pelo tempo do mestre (sem sufixo)

IDIOMAS_STANZA = {"pt": "pt", "en": "en", "es": "es", "fr": "fr", "zh": "zh-hans"}  # idiomas que usam Stanza
                      # "zh-hans" = chinês simplificado (o modelo Stanza correspondente);
                      # troque por "zh-hant" se um dia quiser o tradicional
IDIOMA_KIWI = "ko"  # idioma que usa Kiwi (só coreano, por enquanto)

BOX_BORDER = 6  # espessura da caixa colorida, em px

print(f"Vídeo: {NOME_ORACAO}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. BAIXAR OS SRTs — mestre usa "whisper", os outros usam o          ║
# ║  conjunto JÁ SINCRONIZADO (sem sufixo, gerado pelo                ║
# ║  caption-multilang-generate.ipynb) — não usa mais o "whisper" bruto ║
# ║  dos outros idiomas, que tem timing próprio de cada dublagem e    ║
# ║  fica fora de sincronia por conteúdo.                             ║
# ╚══════════════════════════════════════════════════════════════════╗
from config import PipelineConfig
from drive_utils import DriveClient
from srt_utils import ler_srt

drive_client = DriveClient.get()
legendas_por_idioma_raw = {}

todos_idiomas = list(IDIOMAS_STANZA.keys()) + [IDIOMA_KIWI]
for idioma in todos_idiomas:
    config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE=IDIOMA_MESTRE)

    # mestre = "whisper" (é a própria referência, não passa por sincronização);
    # os outros = sem sufixo (conjunto já sincronizado pelo tempo do mestre)
    if idioma == IDIOMA_MESTRE:
        nome_arquivo = config.nome_srt_whisper(idioma)
    else:
        nome_arquivo = config.nome_srt(idioma)

    destino_local = Path(nome_arquivo)
    ok = drive_client.download(config.pasta_oracao, nome_arquivo, destino_local)
    if not ok:
        print(f"  ⚠️  {idioma.upper()}: não achei '{nome_arquivo}' — pulando")
        continue
    legendas_por_idioma_raw[idioma] = ler_srt(destino_local)
    print(f"  ✅ {idioma.upper()} ({nome_arquivo}): {len(legendas_por_idioma_raw[idioma])} bloco(s)")

if not legendas_por_idioma_raw:
    raise FileNotFoundError("Nenhum SRT encontrado pra nenhum idioma")

if IDIOMA_MESTRE not in legendas_por_idioma_raw:
    print(f"⚠️  O idioma mestre ('{IDIOMA_MESTRE}') não foi encontrado — as legendas dos outros "
          f"idiomas estão sincronizadas em relação a ele, então isso é só um aviso informativo, "
          f"não impede o resto de rodar.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. CLASSIFICAR CADA BLOCO (Stanza pra PT/EN/ES/FR, Kiwi pro KO) ║
# ║  Se já existir uma classificação salva/corrigida no Drive        ║
# ║  (config.nome_classificacao_multicolor), usa ela em vez de rodar ║
# ║  o Stanza/Kiwi de novo pra esse idioma.                          ║
# ╚══════════════════════════════════════════════════════════════════╝
from classificacao import classificar_palavra_stanza
from classificacao_ko import classificar_pecas_palavra_ko
from renderizacao import PecaColorida, salvar_classificacao_multicolor, carregar_classificacao_multicolor

def _tentar_carregar_classificacao_salva(idioma):
    nome_arquivo = config.nome_classificacao_multicolor(idioma)
    destino_local = Path(nome_arquivo)
    if drive_client.download(config.pasta_oracao, nome_arquivo, destino_local):
        return carregar_classificacao_multicolor(destino_local)
    return None

blocos_por_idioma = {}
idiomas_reaproveitados = []
idiomas_a_classificar = []

for idioma in list(IDIOMAS_STANZA.keys()) + [IDIOMA_KIWI]:
    if idioma not in legendas_por_idioma_raw:
        continue
    blocos_salvos = _tentar_carregar_classificacao_salva(idioma)
    if blocos_salvos is not None:
        blocos_por_idioma[idioma] = blocos_salvos
        idiomas_reaproveitados.append(idioma)
        print(f"✅ {idioma.upper()}: classificação já salva reaproveitada "
              f"({config.nome_classificacao_multicolor(idioma)}, {len(blocos_salvos)} bloco(s))")
    else:
        idiomas_a_classificar.append(idioma)

# ── Pipelines Stanza — só pros idiomas que ainda precisam ser classificados ──
pipelines_stanza = {}
for idioma, codigo in IDIOMAS_STANZA.items():
    if idioma not in idiomas_a_classificar:
        continue
    stanza.download(codigo, verbose=False)
    pipelines_stanza[idioma] = stanza.Pipeline(codigo, processors="tokenize,pos,lemma", verbose=False)
    print(f"✅ Pipeline Stanza pronto: {idioma}")

# ── PT / EN / ES / FR — todos iguais, palavra por palavra (a classificação
# ── simplificada não precisa mais de tratamento especial pro francês,
# ── já que substantivo não distingue mais gênero) ──────────────────────
for idioma in IDIOMAS_STANZA:
    if idioma not in idiomas_a_classificar:
        continue
    nlp = pipelines_stanza[idioma]
    blocos = []
    for leg in legendas_por_idioma_raw[idioma]:
        doc = nlp(leg.texto)
        pecas = []
        for sentenca in doc.sentences:
            for palavra in sentenca.words:
                classe = classificar_palavra_stanza(
                    palavra.text, palavra.lemma, palavra.upos, palavra.xpos,
                    palavra.feats or "", idioma,
                )
                pecas.append(PecaColorida(palavra.text, classe))
        blocos.append({"inicio_ms": leg.inicio_ms, "fim_ms": leg.fim_ms, "pecas": pecas})
    blocos_por_idioma[idioma] = blocos

    nome_arquivo = config.nome_classificacao_multicolor(idioma)
    salvar_classificacao_multicolor(blocos, Path(nome_arquivo))
    drive_client.upload(Path(nome_arquivo), config.pasta_oracao, "application/json")
    print(f"✅ {idioma.upper()} classificado: {len(blocos)} bloco(s) (salvo em {nome_arquivo})")

# ── COREANO — peça por peça, com colado_anterior pra não ter espaço dentro
# ── da mesma palavra original ────────────────────────────────────────────
if IDIOMA_KIWI in idiomas_a_classificar:
    blocos = []
    for leg in legendas_por_idioma_raw[IDIOMA_KIWI]:
        resultado = kiwi.analyze(leg.texto)
        tokens = resultado[0][0]
        grupos: dict[tuple, list] = {}
        ordem_grupos = []
        for t in tokens:
            chave = (t.sent_position, t.word_position)
            if chave not in grupos:
                grupos[chave] = []
                ordem_grupos.append(chave)
            grupos[chave].append({"peca": t.form, "classe_kiwi": t.tag})

        pecas = []
        for chave in ordem_grupos:
            pecas_da_palavra = grupos[chave]
            classes = classificar_pecas_palavra_ko(pecas_da_palavra)
            for i, (p, classe) in enumerate(zip(pecas_da_palavra, classes)):
                pecas.append(PecaColorida(p["peca"], classe, colado_anterior=(i > 0)))
        blocos.append({"inicio_ms": leg.inicio_ms, "fim_ms": leg.fim_ms, "pecas": pecas})
    blocos_por_idioma[IDIOMA_KIWI] = blocos

    nome_arquivo = config.nome_classificacao_multicolor(IDIOMA_KIWI)
    salvar_classificacao_multicolor(blocos, Path(nome_arquivo))
    drive_client.upload(Path(nome_arquivo), config.pasta_oracao, "application/json")
    print(f"✅ KO classificado: {len(blocos)} bloco(s) (salvo em {nome_arquivo})")

if idiomas_reaproveitados:
    print(f"\nℹ️  {len(idiomas_reaproveitados)} idioma(s) usaram classificação já salva "
          f"(possivelmente corrigida à mão): {', '.join(i.upper() for i in idiomas_reaproveitados)}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. GERAR O .ASS COM CAIXA COLORIDA                              ║
# ╚══════════════════════════════════════════════════════════════════╝
from renderizacao import gerar_ass

config_render = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE=IDIOMA_MESTRE,
                               SUFIXO_VARIANTE_IDIOMAS="_zh")

caminho_ass = gerar_ass(blocos_por_idioma, config_render, box_border=BOX_BORDER,
                        caminho_saida=Path(f"legendas_{NOME_ORACAO}_zh.ass"))
print(f"✅ Legenda gerada: {caminho_ass}")
print("\nRode a célula 6 pra baixar e revisar. Só rode a célula 7 (salvar no Drive) depois de confirmar que ficou bom.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. BAIXAR O .ASS (pra revisar / corrigir manualmente)           ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import files
files.download(str(caminho_ass))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  7. SALVAR NO DRIVE — só rode depois de conferir que ficou bom   ║
# ║  (revise o .ass baixado na célula 6 antes de rodar essa aqui)    ║
# ╚══════════════════════════════════════════════════════════════════╝
caminho_no_drive = drive_client.upload(caminho_ass, config_render.pasta_oracao)
print(f"✅ Salvo no Drive: {caminho_no_drive}")
